In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np 

In [4]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\SAISH\Desktop\ML Subject Project\ML-MLOps-Project\notebooks
['.gitkeep', 'exp1.ipynb', 'exp2_bow_vs_tfidf.py', 'exp3_lor_bow_hp.py']


In [5]:
import pandas as pd
df = pd.read_csv('../data/raw/IMDB.csv')
print("Dataset shape:", df.shape)
df.to_csv('../data/raw/data.csv', index=False)
df.head()

Dataset shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = text.lower()                               # lower case
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # remove punctuation
    text = re.sub(r'\d+', '', text)                    # remove numbers
    
    words = text.split()
    
    words = [word for word in words if word not in stop_words]  # remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words]      # lemmatization
    
    return " ".join(words)


def normalize_text(df):
    try:
        df['review'] = df['review'].apply(preprocess_text)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
0,one reviewer mentioned watching oz episode hoo...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake think zombie ...,negative
4,petter mattei love time money visually stunnin...,positive


In [8]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df = df.reset_index(drop=True)

In [10]:
print(df['sentiment'].value_counts())
print(df.isnull().sum())

sentiment
1    25000
0    25000
Name: count, dtype: int64
review       0
sentiment    0
dtype: int64


In [11]:
print(df.shape)

(50000, 2)


In [14]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words="english"
)

X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
import dagshub
import mlflow

mlflow.set_tracking_uri(
    "https://dagshub.com/saishkhandekar13/ML-MLOps-Project.mlflow"
)

dagshub.init(
    repo_owner="saishkhandekar13",
    repo_name="ML-MLOps-Project",
    mlflow=True
)

mlflow.set_experiment("Logistic Regression Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\SAISH\miniconda3\envs\atlas\lib\site-packages\rich\live.py:229: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=02c91caf-cb7b-4523-b257-272b01638704&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4d58b1793a4482c05a52812855aae1838af06eb96f3e90e87030f5b1e0b64d2f




Accessing as saishkhandekar13

Repository ML-MLOps-Project doesn't exist, creating it under current user.

Initialized MLflow to track repo "saishkhandekar13/ML-MLOps-Project"

Repository saishkhandekar13/ML-MLOps-Project initialized!

2026/03/15 18:48:28 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5faf82168eae46ec8689180b015d4e09', creation_time=1773580633534, experiment_id='0', last_update_time=1773580633534, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [ ]:
import mlflow
import logging
import time
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():

    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")

        mlflow.log_param("vectorizer", "CountVectorizer")
        mlflow.log_param("max_features", 10000)
        mlflow.log_param("ngram_range", "(1,2)")
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("dataset_size", len(df))

        logging.info("Initializing Logistic Regression model...")

        model = LogisticRegression(
            max_iter=2000,
            solver="liblinear"
        )

        logging.info("Training model...")

        model.fit(X_train, y_train)

        logging.info("Model training completed")

        mlflow.log_param("model", "LogisticRegression")

        logging.info("Running predictions...")

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging metrics...")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)

        mlflow.log_metric("true_positive", cm[1][1])
        mlflow.log_metric("true_negative", cm[0][0])

        # Save vectorizer for inference
        joblib.dump(vectorizer, "vectorizer.pkl")
        mlflow.log_artifact("vectorizer.pkl")

        logging.info("Saving model to MLflow...")

        mlflow.sklearn.log_model(
            model,
            "model",
            input_example=X_test[:5]
        )

        end_time = time.time()

        logging.info(f"Training finished in {end_time-start_time:.2f} seconds")

        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"Error occurred: {e}", exc_info=True)
        

2026-03-15 19:02:52,011 - INFO - Starting MLflow run...
2026-03-15 19:02:53,136 - INFO - Logging preprocessing parameters...
2026-03-15 19:02:54,867 - INFO - Initializing Logistic Regression model...
2026-03-15 19:02:54,868 - INFO - Training model...
2026-03-15 19:03:01,814 - INFO - Model training completed
2026-03-15 19:03:02,159 - INFO - Running predictions...
2026-03-15 19:03:02,182 - INFO - Logging metrics...
2026-03-15 19:03:06,781 - INFO - Saving model to MLflow...
c:\Users\SAISH\miniconda3\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\SAISH\mini